In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers peft accelerate bitsandbytes accelerate

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 76.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 112.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 56.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 91.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [1]:
import pandas as pd

def split(csv_path, train_ratio=0.6, val_ratio=0.2):
    df = pd.read_csv(csv_path)
    df = df.sort_values("Timestamp").reset_index(drop=True)
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    train_df = df.iloc[:train_end]
    val_df = df.iloc[train_end:val_end]
    test_df = df.iloc[val_end:]
    return df,train_df, val_df, test_df


In [2]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch

df,train_df,val_df,test_df = split("/content/BGL_1000k_log_structured_cleaned.csv")
id_to_template = dict(zip(df["EventId"], df["EventTemplate"]))
special_tokens = ["[PAD]", "[CLS]", "[MASK]", "[UNK]"]
event_ids = df["EventId"].unique().tolist()

event2id = {token: i for i, token in enumerate(special_tokens)}
for i, eid in enumerate(event_ids):
    event2id[eid] = i + len(special_tokens)

id2event = {v: k for k, v in event2id.items()}
vocab_size = len(event2id)

print(f"Vocab Size: {vocab_size}")

train_df.head()

Vocab Size: 242


,LineId,Label,Timestamp,Date,Node,Time,NodeRepeat,Type,Component,Level,Content,EventId,EventTemplate
0,4,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.823719,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,E1,instruction cache parity error corrected
1,2,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.527847,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,E1,instruction cache parity error corrected
2,3,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.675872,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,E1,instruction cache parity error corrected
3,1,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.363779,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,E1,instruction cache parity error corrected
4,5,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.982731,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,E1,instruction cache parity error corrected


In [3]:
id_to_template = dict(zip(df["EventId"], df["EventTemplate"]))

In [ ]:
id2event = {v: k for k, v in event2id.items()}

In [4]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WINDOW_SIZE = 128
BATCH_SIZE = 128
EPOCHS = 50
STEP_SIZE = 32

In [5]:
import pandas as pd
from tqdm import tqdm
def generate_sequences(df, window_size, step_size, min_anomaly_check=True):
    df_sorted = df.sort_values("Timestamp").reset_index(drop=True)
    event_ids = df_sorted["EventId"].values
    labels = df_sorted["Label"].values
    sequences  = []
    seq_labels = []
    for i in range(0, len(event_ids) - window_size, step_size):
        seq = event_ids[i : i + window_size].tolist()
        is_anomaly = int(any(l != '-' and l != 0 for l in labels[i : i + window_size]))
        sequences.append(seq)
        seq_labels.append(is_anomaly)

    return sequences, seq_labels

In [6]:
class LogDataset(Dataset):
    def __init__(self, sequences, event2id, seq_labels, mask_ratio=0.15):
        self.sequences  = sequences
        self.event2id   = event2id
        self.labels     = seq_labels
        self.mask_ratio = mask_ratio
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        line    = self.sequences[idx]
        encoded = [self.event2id.get(eid, self.event2id["[UNK]"]) for eid in line]
        tokens  = [self.event2id["[CLS]"]] + encoded
        input_ids    = []
        output_label = []
        for i, token in enumerate(tokens):
            if i == 0:
                input_ids.append(token)
                output_label.append(-100)
                continue
            if np.random.random() < self.mask_ratio:
                p = np.random.random()
                if p < 0.80:
                    input_ids.append(self.event2id["[MASK]"])
                elif p < 0.90:
                    input_ids.append(np.random.randint(
                        len(special_tokens), len(self.event2id)))
                else:
                    input_ids.append(token)
                output_label.append(token)
            else:
                input_ids.append(token)
                output_label.append(-100)
        return (
            torch.tensor(input_ids),
            torch.tensor(output_label),
            torch.tensor(tokens),
        )


In [7]:
all_seqs, all_labels = generate_sequences(df, WINDOW_SIZE, STEP_SIZE)
n = len(all_seqs)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_seqs = all_seqs[:train_end]
train_labels = all_labels[:train_end]
val_seqs = all_seqs[train_end:val_end]
val_labels = all_labels[train_end:val_end]
test_seqs = all_seqs[val_end:]
test_gt = all_labels[val_end:]

train_normal_seqs = [s for s, l in zip(train_seqs, train_labels) if l == 0]
val_normal_seqs = [s for s, l in zip(val_seqs,   val_labels)   if l == 0]

train_dataset = LogDataset(train_normal_seqs,event2id, [],mask_ratio=0.15)
val_dataset = LogDataset(val_normal_seqs,event2id, [],mask_ratio=0.15)
test_dataset = LogDataset(test_seqs,event2id, test_gt,mask_ratio=0.15)

train_loader  = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset,  batch_size=128, shuffle=False)

print(f"test_gt length      : {len(test_gt)}")
print(f"test_dataset length : {len(test_dataset)}")
print(f"test_loader batches : {len(test_loader)}")

test_gt length      : 6250
test_dataset length : 6250
test_loader batches : 49


In [8]:
import torch
import torch.nn as nn
class LogBERT(nn.Module):
    def __init__(self, vocab_size, hidden_size=256, num_layers=2,
                 num_heads=4, max_len=514, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding     = nn.Embedding(vocab_size, hidden_size)
        self.pos_embedding = nn.Parameter(
            torch.randn(1, max_len, hidden_size) * 0.02
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=num_heads,
            batch_first=True, dropout=dropout,
            dim_feedforward=128
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.mask_lm = nn.Linear(hidden_size, vocab_size)
        self._init_weights()
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0, std=0.02)
    def forward(self, x):
        b, seq_len = x.size()
        x       = self.embedding(x) + self.pos_embedding[:, :seq_len, :]
        encoded = self.encoder(x)
        logits  = self.mask_lm(encoded)
        cls_emb = encoded[:, 0, :]
        return logits, cls_emb

In [9]:
model = LogBERT(len(event2id), hidden_size=256, num_layers=2,
                    num_heads=4, max_len=514).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
print(f"\nModel parameters : {sum(p.numel() for p in model.parameters()):,}")


Model parameters : 915,954


In [10]:
from tqdm import tqdm
import math
alpha        = 0.0
best_val_loss = float('inf')
WARMUP    = 2
def get_lr_scale(epoch, warmup_epochs=WARMUP):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, EPOCHS - warmup_epochs)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler     = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr_scale)
best_val_loss = float('inf')
for epoch in tqdm(range(EPOCHS)):
    model.train()
    train_loss = 0.0
    total_masked = 0
    total_correct = 0
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, _ = model(x)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        train_loss += loss.item()
        with torch.no_grad():
            mask = y != -100
            preds = logits.argmax(dim=-1)
            correct = (preds[mask] == y[mask]).sum().item()
            total_correct += correct
            total_masked  += mask.sum().item()

    scheduler.step()
    train_acc = total_correct / total_masked * 100 if total_masked > 0 else 0
    model.eval()
    val_loss    = 0.0
    val_masked  = 0
    val_correct = 0

    with torch.no_grad():
        for x, y, _ in val_loader:
            x, y  = x.to(DEVICE), y.to(DEVICE)
            logits, _ = model(x)
            val_loss += criterion(
                logits.view(-1, logits.size(-1)), y.view(-1)).item()

            mask = y != -100
            preds = logits.argmax(dim=-1)
            val_correct += (preds[mask] == y[mask]).sum().item()
            val_masked  += mask.sum().item()

    avg_train = train_loss  / len(train_loader)
    avg_val = val_loss    / len(val_loader)
    val_acc = val_correct / val_masked * 100 if val_masked > 0 else 0
    lr_now  = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {avg_train:.4f} Acc: {train_acc:.1f}% | "
          f"Val Loss: {avg_val:.4f} Acc: {val_acc:.1f}% | "
          f"LR: {lr_now:.6f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "fn_logbert.pt")
        print(f"Best model saved")

  2%|▏         | 1/50 [00:07<06:29,  7.94s/it]

Epoch 01 | Train Loss: 2.0399 Acc: 65.1% | Val Loss: 4.5791 Acc: 24.2% | LR: 0.000100
Best model saved


  4%|▍         | 2/50 [00:15<06:16,  7.85s/it]

Epoch 02 | Train Loss: 0.3631 Acc: 88.6% | Val Loss: 4.0857 Acc: 34.2% | LR: 0.000100
Best model saved


  6%|▌         | 3/50 [00:24<06:19,  8.08s/it]

...

Epoch 48 | Train Loss: 0.1682 Acc: 90.4% | Val Loss: 2.0447 Acc: 64.6% | LR: 0.000010


 98%|█████████▊| 49/50 [06:38<00:08,  8.19s/it]

Epoch 49 | Train Loss: 0.1666 Acc: 90.5% | Val Loss: 2.0344 Acc: 64.7% | LR: 0.000010


100%|██████████| 50/50 [06:46<00:00,  8.12s/it]

Epoch 50 | Train Loss: 0.1680 Acc: 90.5% | Val Loss: 2.0590 Acc: 64.4% | LR: 0.000010


In [11]:
model_test = LogBERT(len(event2id), hidden_size=256, num_layers=2,
                     num_heads=4, max_len=514).to(DEVICE)
model_test.load_state_dict(torch.load("fn_logbert.pt", weights_only=True))
model_test.eval()

LogBERT(
  (embedding): Embedding(242, 256)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=128, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=128, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (mask_lm): Linear(in_features=256, out_features=242, bias=True)
)

In [ ]:
print("Test sequences:", len(test_dataset))
print("Test batches:", len(test_loader))

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

g_candidates = [3, 5, 9, 15, 20]
val_counts_by_g = {}
model_test.eval()
with torch.no_grad():
    for g_val in g_candidates:
        val_counts = []

        for x, y, original_ids in val_loader:
            x = x.to(DEVICE)
            logits, _ = model_test(x)
            probs = torch.softmax(logits, dim=-1)
            top_g_idx = torch.topk(probs, g_val, dim=-1).indices.cpu()
            y_cpu = y.cpu()
            orig_cpu = original_ids.cpu()

            for i in range(x.size(0)):
                wrong = 0
                total_masked = 0
                for j in range(1, y_cpu[i].size(0)):
                    if y_cpu[i][j].item() == -100:
                        continue
                    total_masked += 1
                    if orig_cpu[i][j].item() not in top_g_idx[i, j].tolist():
                        wrong += 1
                val_counts.append(wrong)

        val_counts_by_g[g_val] = np.array(val_counts)
print(f"\n{'g':>4} | wrong count distribution on val (all normal)")
print(f"{'':>4}   50th  90th  95th  99th  → chosen r")
print("-" * 52)
chosen_r = {}
for g_val in g_candidates:
    counts   = val_counts_by_g[g_val]
    p50 = np.percentile(counts, 50)
    p90 = np.percentile(counts, 90)
    p95 = np.percentile(counts, 95)
    p99 = np.percentile(counts, 99)
    r_chosen = int(p99)
    chosen_r[g_val] = r_chosen
    print(f"g={g_val:>2} | {p50:>5.1f} {p90:>5.1f} {p95:>5.1f} {p99:>5.1f}  → r={r_chosen}")


   g | wrong count distribution on val (all normal)
       50th  90th  95th  99th  → chosen r
----------------------------------------------------
g= 3 |   0.0  10.0  12.0  15.0  → r=15
g= 5 |   0.0   9.0  12.0  15.0  → r=15
g= 9 |   0.0   9.0  11.0  15.0  → r=15
g=15 |   0.0   9.0  11.0  14.0  → r=14
g=20 |   0.0   8.0  11.0  14.0  → r=14


In [58]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
g = 5
r = 15
pred_labels = []
with torch.no_grad():
    for x, _, original_ids in test_loader:
        x = x.to(DEVICE)
        logits, _ = model_test(x)
        probs = torch.softmax(logits, dim=-1)
        top_g_idx = torch.topk(probs, g, dim=-1).indices.cpu()
        orig_cpu = original_ids.cpu()
        for i in range(x.size(0)):
            count = 0
            for j in range(1, orig_cpu[i].size(0)):
                if orig_cpu[i][j].item() not in top_g_idx[i, j].tolist():
                    count += 1
            pred_labels.append(1 if count > r else 0)
p   = precision_score(test_gt, pred_labels, zero_division=0)
r_  = recall_score(test_gt, pred_labels, zero_division=0)
f1  = f1_score(test_gt, pred_labels, zero_division=0)
print(f"Precision : {p:.4f}")
print(f"Recall    : {r_:.4f}")
print(f"F1        : {f1:.4f}")

Precision : 0.8794
Recall    : 0.8663
F1        : 0.8728


Proving val set is not distributed properly

In [56]:
print(f"Train anomaly rate: {sum(train_labels)/len(train_labels):.3f}")
print(f"Val anomaly rate  : {sum(val_labels)/len(val_labels):.3f}")
print(f"Test anomaly rate : {sum(test_gt)/len(test_gt):.3f}")

Train anomaly rate: 0.435
Val anomaly rate  : 0.021
Test anomaly rate : 0.032


In [21]:
anomalies    = []
g = 5
with torch.no_grad():
    for batch_idx, (x, y, original_ids) in enumerate(test_loader):
        x = x.to(DEVICE)
        logits, _ = model_test(x)
        probs = torch.softmax(logits, dim=-1)
        top_g_idx = torch.topk(probs, g, dim=-1).indices.cpu()
        y_cpu = y.cpu()
        orig_cpu = original_ids.cpu()
        for i in range(x.size(0)):
            wrong        = 0
            total_masked = 0
            for j in range(1, y_cpu[i].size(0)):
                if y_cpu[i][j].item() == -100:
                    continue
                total_masked += 1
                real_token    = orig_cpu[i][j].item()
                if real_token not in top_g_idx[i, j].tolist():
                    wrong += 1
            if total_masked == 0:
                continue

            global_seq_id  = batch_idx * test_loader.batch_size + i
            if wrong > r:
                orig_tokens = orig_cpu[i].tolist()
                event_ids   = [id2event[t] for t in orig_tokens if t not in (event2id["[CLS]"],event2id["[PAD]"])]
                anomalies.append({
                    "sequence_id"  : global_seq_id,
                    "anomaly_score": round(wrong, 4),
                    "events"       : event_ids
                })
print(f"Anomalies detected : {len(anomalies)}")

Anomalies detected : 100


In [22]:
for anomaly in anomalies:
    anomaly["events"] = [
        id_to_template[eid] for eid in anomaly["events"]
    ]

In [25]:
import json
with open("/content/anomalies5.json", "w") as f:
    json.dump(anomalies, f, indent=2)

In [29]:
def format_instruction(sample):
    instruction = "Analyze the following log sequence and determine if it is normal or abnormal. Provide a concise explanation for your judgment."
    input_text = f"Input: {sample['log_seq']}"
    output_text = f"Output: {sample['label-cause pair']}"
    return f"### Instruction:\n{instruction}\n\n### {input_text}\n\n### {output_text}"

In [27]:
from google.colab import userdata
secret_value_0 = userdata.get('HF_TOKEN')

In [28]:
import os
from huggingface_hub import login

login(secret_value_0)

### Fine Tuning LLMs

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

In [30]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "meta-llama/Meta-Llama-3-8B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [31]:
from transformers import Trainer, TrainingArguments
import torch

In [32]:
import ast
with open('/content/seed_bgl.json', 'r') as f:
    raw_data = ast.literal_eval(f.read())
formatted_data = [format_instruction(item) for item in raw_data]
from datasets import Dataset
dataset = Dataset.from_dict({"text": formatted_data})
print(dataset[0]['text'])

### Instruction:
Analyze the following log sequence and determine if it is normal or abnormal. Provide a concise explanation for your judgment.

### Input: quiet <*>................................<*>,minus inf................................<*>,minus normalized number..................<*>,minus denormalized number................<*>,plus zero................................<*>,plus denormalized number.................<*>,plus normalized number...................<*>,plus infinity............................<*>,reserved.................................<*>,invalid operation exception (software)...<*>,invalid operation exception (sqrt).......<*>,invalid operation exception (int cnvt)...<*>,enable invalid operation exceptions......<*>,enable overflow exceptions...............<*>,enable underflow exceptions..............<*>,enable divide-by-zero exceptions.........<*>,enable inexact exceptions................<*>,enable non-IEEE mode.....................<*>,round nearest.....................

In [33]:
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [34]:
import torch.nn as nn
class LLMLADELoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.ce_loss = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, labels):
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        logits_flat = shift_logits.view(-1, shift_logits.size(-1))
        labels_flat = shift_labels.view(-1)
        raw_loss = self.ce_loss(logits_flat, labels_flat)
        probs = torch.softmax(logits_flat, dim=-1)
        pt = probs.gather(1, labels_flat.unsqueeze(1)).squeeze(1)
        focal_loss = (self.alpha * (1 - pt)**self.gamma * raw_loss)
        cause_loss = (self.beta * raw_loss)
        return focal_loss.mean() + cause_loss.mean()

In [35]:
class LLMLADETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False,**kwargs):
        outputs = model(**inputs)
        logits = outputs.get("logits")
        labels = inputs.get("labels")

        loss_fct = LLMLADELoss(alpha=0.7, beta=0.3, gamma=2.0)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [36]:
training_args = TrainingArguments(
    output_dir="./llm_lade_results",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=True,
    logging_steps=5,
)
trainer = LLMLADETrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Step,Training Loss
5,14.946945
10,7.193362
15,5.556383
20,4.792670
25,4.732492
30,4.204222
35,4.712759
40,3.967097


TrainOutput(global_step=40, training_loss=6.263241338729858, metrics={'train_runtime': 571.6902, 'train_samples_per_second': 0.525, 'train_steps_per_second': 0.07, 'total_flos': 6929101357056000.0, 'train_loss': 6.263241338729858, 'epoch': 10.0})

In [38]:
trainer.save_model("./llm_lade_results")

In [ ]:
!zip -r llm_lade_results.zip llm_lade_results

In [39]:
input_file = "/content/anomalies5.json"
output_file = "/content/llm_results.json"

In [40]:
import torch
import gc
if 'model' in locals(): del model
if 'base_model' in locals(): del base_model
if 'trainer' in locals(): del trainer
gc.collect()
torch.cuda.empty_cache()
free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
print(f"Free GPU memory: {free_memory / 1024**3:.2f} GB")

Free GPU memory: 14.49 GB


In [41]:
import torch
from peft import PeftModel
import os
base_model_path = "meta-llama/Meta-Llama-3-8B"
adapter_path = os.path.abspath("./llm_lade_results")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
tokenizer.pad_token = tokenizer.eos_token
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B",
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [42]:
import json
with open(input_file, 'r') as f:
    anomalies = json.load(f)

In [43]:
from collections import Counter
def compress_sequence(events, max_unique=20, max_total=40):
    counts = Counter(events)
    compressed = []
    seen = set()
    for event in events:
        if event not in seen:
            seen.add(event)
            count = counts[event]
            if count > 1:
                compressed.append(f"[x{count}] {event}")
            else:
                compressed.append(event)
        if len(compressed) >= max_unique:
            break
    return compressed

In [46]:
from tqdm import tqdm
finals = []
with torch.inference_mode():
    for entry in tqdm(anomalies):
        seq = compress_sequence(entry['events'])
        log_seq_str = "\n".join(seq)
        instruction = (
            "Analyze the following log sequence. Determine if it is normal or abnormal. "
            "Provide a concise explanation of the root cause if abnormal, "
            "and suggest a specific, actionable solution to resolve the issue."
        )
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{log_seq_str}\n\n"
            f"### Output:\nVerdict: "
        )
        inputs = tokenizer(prompt, return_tensors="pt", max_length=512,truncation=True).to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=0.0
        )
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        verdict = "Verdict: " + full_output.split("### Output:")[-1].replace("Verdict: ", "").strip()
        finals.append({
            "seq_id": entry["sequence_id"],
            "log_seq": log_seq_str,
            "anomaly_score": entry["anomaly_score"],
            "llm_verdict": verdict
        })
        gc.collect()
        torch.cuda.empty_cache()

  0%|          | 0/100 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  1%|          | 1/100 [00:25<41:30, 25.15s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  2%|▏         | 2/100 [00:47<38:34, 23.62s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentati

In [47]:
with open(output_file, 'w') as f:
    json.dump(finals, f, indent=4)